In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder. \
    appName("Himanshu"). \
    getOrCreate()

In [3]:
spark.conf.set("spark.debug.maxToStringFields",2000)

### Read data

In [4]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

### Sample function

In [5]:
from pyspark.sql import DataFrame
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [6]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [7]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result

In [8]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Data Exploration - KPIs

In [9]:
df.show(5)

+------+--------------------+------------+--------------+--------------------+--------------------+-------------+-----+--------------------+-----------------------------+-----------------+---------------+----------------+--------------------+--------------------+--------------------+-------------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+
|Job ID|              Agency|Posting Type|# Of Positions|      Business Title| Civil Service Title|Title Code No|Level|        Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|       Work Location|  Division/Work Unit|     Job Description|Minimum Qual Requirements|    Preferred Skills|Additional Information|            To Apply|         Hours/Shift|     Work Location 1| Recruitment Contact|Residency Require

In [10]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

df.printSchema()

Total Rows: 2946
Total Columns: 28
root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string 

### Identify categorical vs numerical columns

In [11]:
categorical_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]
numeric_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, (IntegerType, DoubleType))]

print("Categorical Columns:", categorical_columns)
print("Numeric Columns:", numeric_columns)

Categorical Columns: ['Job ID', 'Agency', 'Posting Type', '# Of Positions', 'Business Title', 'Civil Service Title', 'Title Code No', 'Level', 'Job Category', 'Full-Time/Part-Time indicator', 'Salary Range From', 'Salary Range To', 'Salary Frequency', 'Work Location', 'Division/Work Unit', 'Job Description', 'Minimum Qual Requirements', 'Preferred Skills', 'Additional Information', 'To Apply', 'Hours/Shift', 'Work Location 1', 'Recruitment Contact', 'Residency Requirement', 'Posting Date', 'Post Until', 'Posting Updated', 'Process Date']
Numeric Columns: []


### Average salary column

In [12]:
df = df.withColumn(
    "avg_salary",
    (col("Salary Range From") + col("Salary Range To")) / 2
)

df.select("Salary Range From","Salary Range To","avg_salary").show(5)

+-----------------+---------------+----------+
|Salary Range From|Salary Range To|avg_salary|
+-----------------+---------------+----------+
|            42405|          65485|   53945.0|
|            60740|         162014|  111377.0|
|         51907.68|       54580.32|   53244.0|
|         51907.68|       54580.32|   53244.0|
|               35|             35|      35.0|
+-----------------+---------------+----------+
only showing top 5 rows



### KPI 1 — Top 10 job categories by postings

In [13]:
kpi1 = df.groupBy("Job Category") \
    .count() \
    .orderBy(desc("count")) \
    .limit(10)

kpi1.show()

+--------------------+-----+
|        Job Category|count|
+--------------------+-----+
|Engineering, Arch...|  504|
|Technology, Data ...|  313|
|       Legal Affairs|  226|
|Public Safety, In...|  182|
|Building Operatio...|  181|
|Finance, Accounti...|  169|
|Administration & ...|  134|
|Constituent Servi...|  129|
|              Health|  125|
|Policy, Research ...|  124|
+--------------------+-----+



In [ ]:
kpi1.toPandas().plot(
    kind="bar",
    x="Job Category",
    y="count",
    title="Top 10 Job Categories"
)

### KPI 2 — Salary distribution per job category

In [14]:
kpi2 = df.groupBy("Job Category") \
    .agg(
        avg("avg_salary").alias("average_salary"),
        max("avg_salary").alias("max_salary"),
        min("avg_salary").alias("min_salary")
    ) \
    .orderBy(desc("average_salary"))

kpi2.show()

+--------------------+------------------+----------+----------+
|        Job Category|    average_salary|max_salary|min_salary|
+--------------------+------------------+----------+----------+
|Administration & ...|          218587.0|  218587.0|  218587.0|
|Engineering, Arch...|          198518.0|  198518.0|  198518.0|
|Engineering, Arch...|          196042.5|  209585.0|  182500.0|
|Health Policy, Re...|          128694.5|  162500.0|   94889.0|
|Engineering, Arch...|          128247.5|  128247.5|  128247.5|
|Engineering, Arch...|          128247.5|  128247.5|  128247.5|
|Communications & ...|          125000.0|  125000.0|  125000.0|
|Administration & ...|          118287.0|  118287.0|  118287.0|
|Constituent Servi...|116900.33333333333|  217201.0|   58500.0|
|Constituent Servi...|          103680.5|  103680.5|  103680.5|
|Technology, Data ...|          96373.45|  112527.5|   39119.0|
|Administration & ...|           95000.0|   95000.0|   95000.0|
|Administration & ...|           92500.0

### KPI 3 — Correlation between degree requirement and salary

In [ ]:
df = df.withColumn(
    "requires_degree",
    when(lower(col("Minimum Qual Requirements")).contains("degree"),1).otherwise(0)
)

df.select("requires_degree","avg_salary").show(5)

In [ ]:
correlation = df.stat.corr("requires_degree","avg_salary")

print("Correlation between degree requirement and salary:", correlation)

### KPI 4 — Highest salary job per agency

In [ ]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("Agency").orderBy(desc("avg_salary"))

kpi4 = df.withColumn("rank",row_number().over(windowSpec)) \
    .filter(col("rank")==1) \
    .select("Agency","Business Title","avg_salary")

kpi4.show()

### KPI 5 — Average salary per agency for last 2 years

In [ ]:
df = df.withColumn("Posting Date",to_date(col("Posting Date")))

In [ ]:
recent_jobs = df.filter(
    col("Posting Date") >= add_months(current_date(),-24)
)

In [ ]:
kpi5 = recent_jobs.groupBy("Agency") \
    .agg(avg("avg_salary").alias("average_salary_last_2_years")) \
    .orderBy(desc("average_salary_last_2_years"))

kpi5.show()

### KPI 6 — Highest paid skills

In [ ]:
skills = ["python","sql","aws","spark","java","tableau","excel"]

for skill in skills:
    df = df.withColumn(skill, lower(col("Preferred Skills")).contains(skill))

In [ ]:
skill_salary = []

for skill in skills:
    result = df.filter(col(skill)==True) \
        .agg(avg("avg_salary")) \
        .collect()[0][0]

    skill_salary.append((skill,result))

spark.createDataFrame(skill_salary,["Skill","Average Salary"]).orderBy(desc("Average Salary")).show()

### Save processed dataset

In [ ]:
df.write \
    .mode("overwrite") \
    .parquet("processed_nyc_jobs")

In [ ]:
processed_df = spark.read.parquet("processed_nyc_jobs")

processed_df.show(5)